# **Dự Đoán Chất Lượng Không Khí Cho Ngày Hôm Sau Ở TP.HCM**

Chất lượng không khí tại TP. Hồ Chí Minh ngày càng trở thành mối quan tâm lớn của người dân, đặc biệt với chỉ số AQI trung bình năm 2022-2026 đạt 81.6 (mức Trung bình theo thang US), trong đó nhiều ngày vượt ngưỡng 100 - mức Không tốt cho nhóm nhạy cảm. Khả năng dự báo AQI trước một ngày giúp người dân lên kế hoạch sinh hoạt, hạn chế phơi nhiễm, và hỗ trợ các cơ quan quản lý môi trường.

### **Mục tiêu notebook này:**

- Làm sạch dữ liệu: Xử lý missing values và outliers.
- Tạo features từ chuỗi thời gian: Lag, rolling, đặc trưng thời gian, sin/cos.
- Tạo biến mục tiêu cho cả Regression và Classification.
- Chia tập Train/Test theo thời gian (không shuffle).
- Chuẩn hóa dữ liệu bằng RobustScaler.
- Lưu các file cần thiết cho huấn luyện mô hình: `train.csv`, `test.csv`, `scaler.pkl`, `feature_cols.pkl`.

## **00. Import và cấu hình**

In [ ]:
# Standard Libraries
import warnings
import gdown
import os
import pickle
from google.colab import drive

# Data Manipulation - Math
import numpy as np
import pandas as pd

# Data Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import folium
from matplotlib.colors import LinearSegmentedColormap
from statsmodels.tsa.seasonal import seasonal_decompose
from scipy.stats import gaussian_kde
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.colorbar import ColorbarBase

# Machine Learning - Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

In [ ]:
# Cấu hình
warnings.filterwarnings(
    "ignore"
)  # Tắt các cảnh báo không cần thiết cho notebook sạch hơn
pd.set_option(
    "display.float_format", "{:.2f}".format
)  # Hiển thị số thập phân gọn hơn (làm tròn 2 chữ số thập phân)
plt.rcParams.update(
    {
        "figure.dpi": 150,
        "axes.grid": True,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.family": "DejaVu Sans",
    }
)  # Cấu hình style mặc định cho toàn bộ biểu đồ matplotlib trong notebook

# AQI Color Palette
# Đây là màu chính thức của US EPA, để đảm bảo nhất quán toàn bộ notebook
AQI_COLORS = {
    "Good": "#00E400",
    "Moderate": "#FFFF00",
    "Unhealthy for Sensitive Groups": "#FF7E00",
    "Unhealthy": "#FF0000",
    "Very Unhealthy": "#8F3F97",
    "Hazardous": "#7E0023",
}

AQI_BINS = [0, 50, 100, 150, 200, 300, 500]
AQI_LABELS = list(AQI_COLORS.keys())
AQI_MAPPING = {label: idx for idx, label in enumerate(AQI_LABELS)}


# Hàm AQI_CATEGORY()
def AQI_CATEGORY(value):
    """
    Phân loại mức AQI theo thang US EPA
    Input : Giá trị AQI (số thực)
    Output: Tên mức (string)
    """
    if value <= 50:
        return "Good"
    elif value <= 100:
        return "Moderate"
    elif value <= 150:
        return "Unhealthy for Sensitive Groups"
    elif value <= 200:
        return "Unhealthy"
    elif value <= 300:
        return "Very Unhealthy"
    else:
        return "Hazardous"


# Tạo Custom Gradient Colormap từ AQI_COLORS và AQI_BINS
# Chuẩn hóa các mốc AQI vì LinearSegmentedColormap là thang đo 0.0 đến 1.0
MAX_AQI = 500
positions = [
    val / MAX_AQI for val in AQI_BINS
]  # positions sẽ trở thành [0.0, 0.1, 0.2, 0.3, 0.4, 0.6, 1.0]

# Vì AQI_BINS có 7 mốc nhưng AQI_COLORS chỉ có 6 màu
colors = list(AQI_COLORS.values())
gradient_colors = colors + [colors[-1]]  # Nhân đôi màu cuối cùng

# Ghép vị trí và màu sắc lại để tạo dải gradient
color_mapping = list(
    zip(positions, gradient_colors)
)  # Gán các màu tương ứng với từng giá trị
AQI_gradient_cmap = LinearSegmentedColormap.from_list(
    "AQI_gradient", color_mapping
)  # Tạo ra dãy màu liên tục thay vì riêng lẻ

In [ ]:
# drive.mount('/content/drive')
ROOT = "/content/drive/MyDrive/HCMUS/Nhập Môn Khoa Học Dữ Liệu/Mini Project"

figures_dir = os.path.join(ROOT, "figures")
preprocessing_dir = os.path.join(figures_dir, "preprocessing")
os.makedirs(figures_dir, exist_ok=True)
os.makedirs(preprocessing_dir, exist_ok=True)

In [ ]:
folder_ID = "1_5C0jQ9fI_ALET0DJPypVJeKS4Cxjtt7"
folder_url = f"https://drive.google.com/drive/folders/{folder_ID}"

gdown.download_folder(folder_url, output="data", quiet=False)

df = pd.read_csv("data/air_quality_historical.csv", parse_dates=["date"])
df_air_quality_historical = df.set_index("date").asfreq("D")

df_air_quality_historical.head()

## **02. Tiền Xử Lý (Preprocessing)**

#### **Cấu trúc notebook:**
1. Xử lý dữ liệu và outliers
2. Lag Features - Rolling Features
3. Đặc trưng thời gian
4. Biến mục tiêu
5. Chia tập Train và Test
6. Xử lý outliers trên tập Train bằng Winsorize
7. Chuẩn hóa dữ liệu
8. Lưu file
9. Tổng kết


### **2.1. Xử lý dữ liệu và outliers**

#### **Bỏ eurpean_aqi vì đã sử dụng us_aqi**

Dự án dùng `us_aqi` làm biến mục tiêu theo chuẩn US EPA - phổ biến hơn tại Việt Nam và dễ so sánh với IQAir, AQICN. Cột `european_aqi` không đưa vào model.

In [ ]:
df_air_quality_historical.drop(columns=["european_aqi"], inplace=True)
df_air_quality_historical.head()

#### **Xử lý missing values**

In [ ]:
df_air_quality_historical.isnull().sum()

In [ ]:
df_air_quality_historical[df_air_quality_historical.isnull().any(axis=1)]

In [ ]:
# Điền bù vẫn nên được ưu tiên hơn so với loại bỏ với dữ liệu mang tính thời gian cho dù nó có là những dữ liệu đầu tiên
df_air_quality_historical.interpolate(
    method="time", limit_direction="both", inplace=True
)
df_air_quality_historical.isnull().sum()

In [ ]:
df_air_quality_historical.head()

#### **Kiểm tra outliers**

In [ ]:
cols = df_air_quality_historical.select_dtypes(include="number").columns.tolist()
n = len(cols)
nrows = (n + 4) // 5

plt.figure(figsize=(16, 4 * nrows))
for i, col in enumerate(cols, 1):
    plt.subplot(nrows, 5, i)
    sns.boxplot(y=df_air_quality_historical[col], color="#AED6F1")
    plt.title(col, fontsize=9)
    plt.ylabel("")

plt.suptitle("Boxplot Các Chỉ Số - Kiểm Tra Outliers Trước Xử Lý", fontweight="bold")
plt.tight_layout()
file_path = os.path.join(preprocessing_dir, "boxplot_check_outliers")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()

In [ ]:
numeric_cols = df_air_quality_historical.select_dtypes(
    include=["float64", "int64"]
).columns
outlier_data = []
N = len(df_air_quality_historical)

for col in numeric_cols:
    Q1 = df_air_quality_historical[col].quantile(0.25)
    Q3 = df_air_quality_historical[col].quantile(0.75)
    IQR = Q3 - Q1
    # IQR×1.5: Ngưỡng chuẩn thống kê - dùng để PHÁT HIỆN và báo cáo tỷ lệ
    n_15 = (
        (df_air_quality_historical[col] < Q1 - 1.5 * IQR)
        | (df_air_quality_historical[col] > Q3 + 1.5 * IQR)
    ).sum()
    # IQR×3.0: Ngưỡng conservative - dùng để XỬ LÝ, tránh mất nhiều dữ liệu
    n_30 = (
        (df_air_quality_historical[col] < Q1 - 3.0 * IQR)
        | (df_air_quality_historical[col] > Q3 + 3.0 * IQR)
    ).sum()

    outlier_data.append(
        {
            "Chỉ số": col,
            "Q1": round(Q1, 2),
            "Q3": round(Q3, 2),
            "Outliers (IQR×1.5)": f"{n_15} ({n_15 / N * 100:.1f}%)",
            "Outliers (IQR×3.0)": f"{n_30} ({n_30 / N * 100:.1f}%)",
        }
    )

df_outlier_summary = pd.DataFrame(outlier_data).sort_values(
    "Outliers (IQR×1.5)", ascending=False
)
display(df_outlier_summary.style.hide(axis="index"))

> **Quyết định:** Winsorize dùng ngưỡng `IQR*3.0` vì các ngày AQI cao (130-158) là sự kiện ô nhiễm thật, không phải lỗi đo lường. Xử lý sẽ được thực hiện sau khi chia Train/Test - tránh **data leakage** từ tập Test vào quá trình tính ngưỡng.

### **2.2. Lag Features - Rolling Features**

Model cần dự đoán AQI ngày mai. Nhưng nếu chỉ đưa vào các chỉ số đo ngày hôm nay (pm2_5, pm10, ozone...) thì model không biết xu hướng đang đi lên hay đi xuống nên Lag và Rolling là cách đưa lịch sử vào làm feature để model có ngữ cảnh.

In [ ]:
df_air_quality_historical.head()

#### **Tạo Lag Features: us_aqi tại t-1, t-2, t-3, t-7, t-14**

Trả lời cho câu hỏi "Hôm qua là bao nhiêu?"
- t-1: AQI của ngày hôm qua
- t-7: AQI của tuần trước cùng ngày

In [ ]:
lags = [1, 2, 3, 7, 14]

for lag in lags:
    df_air_quality_historical[f"t-{lag}"] = df_air_quality_historical["us_aqi"].shift(
        lag
    )

df_air_quality_historical[[f"t-{l}" for l in lags]].head(5)

**Dự đoán:** `t-1` sẽ có tương quan cao nhất với TARGET vì điều kiện khí quyển thay đổi chậm trong 24h. `t-7` nắm bắt pattern tuần. NaN ở đầu chuỗi (do shift) sẽ được xử lý khi tạo TARGET.

#### **Tạo Rolling mean/std/max/min: Window = 3, 7, 14, 30 ngày**

Trả lời câu hỏi "Xu hướng gần đây thế nào?"
- rolling_std cho model biết mức độ tin cậy của dự đoán - std thấp thì chuỗi đang ổn định, std cao thì đang bất thường

In [ ]:
windows = [3, 7, 14, 30]  # rolling_1 và rolling_2 vô nghĩa
functions = ["mean", "std", "max", "min"]

for w in windows:
    shifted = df_air_quality_historical["us_aqi"].shift(
        1
    )  # shift(1) trước để tránh data leakage
    # Nếu không shift: rolling(7).mean() tại ngày T sẽ bao gồm giá trị ngày T
    rolling_obj = shifted.rolling(window=w)
    for func in functions:
        df_air_quality_historical[f"rolling_{func}_{w}"] = getattr(rolling_obj, func)()

**Dự đoán:** `rolling_mean_7` tóm tắt xu hướng tuần gần nhất - thường nằm trong top 3 feature quan trọng theo SHAP. `rolling_std_7` đo độ biến động: std cao → ngày hôm nay khó dự đoán hơn. `shift(1)` đảm bảo không có data leakage.

### **2.3. Đặc trưng thời gian**

#### **Tạo sai phân: diff(1) và diff(7) cho AQI**

Thuật ngữ sai phân (differencing) được sử dụng để tính khoảng cách giữa giá trị hiện tại và một giá trị trong quá khứ. Mục đích chính là để làm cho chuỗi dữ liệu trở nên "dừng" (stationary) - tức là loại bỏ các xu hướng (trend) hoặc tính mùa vụ (seasonality) để phục vụ cho việc dự báo.

- **diff(1):** Nó cho biết tốc độ thay đổi của chất lượng không khí từ ngày này sang ngày khác. Nếu diff(1) dương, không khí đang ô nhiễm hơn so với hôm qua. Nếu âm, không khí đang trong lành hơn.

- **diff(7):** Nó so sánh mức độ ô nhiễm của thứ Hai tuần này với thứ Hai tuần trước, thứ Ba tuần này với thứ Ba tuần trước, v.v.

In [ ]:
# 1. Tạo sai phân 1 ngày cho chỉ số us_aqi
df_air_quality_historical["diff_1"] = df_air_quality_historical["us_aqi"].diff(
    periods=1
)

# 2. Tạo sai phân 7 ngày cho chỉ số us_aqi
df_air_quality_historical["diff_7"] = df_air_quality_historical["us_aqi"].diff(
    periods=7
)

#### **Tạo đặc trưng thời gian**

In [ ]:
month = df_air_quality_historical.index.month
weekday = df_air_quality_historical.index.dayofweek
day_of_year = df_air_quality_historical.index.dayofyear

In [ ]:
# Mã hóa tuần hoàn sin/cos (Cyclical Encoding) là cách biến các biến thời gian có tính chu kỳ như:
## Tháng (1 → 12)
## Ngày trong tuần (Thứ 2 → Chủ nhật)
## Ngày trong năm (1 → 365)
# Thành 2 đặc trưng mới bằng hàm sin và cos để mô hình hiểu được tính tuần hoàn

# 1. Month
df_air_quality_historical["month_sin"] = np.sin(2 * np.pi * month / 12)
df_air_quality_historical["month_cos"] = np.cos(2 * np.pi * month / 12)

# 2. Weekday
df_air_quality_historical["weekday_sin"] = np.sin(2 * np.pi * weekday / 7)
df_air_quality_historical["weekday_cos"] = np.cos(2 * np.pi * weekday / 7)

# 3. Day of year
df_air_quality_historical["day_of_year_sin"] = np.sin(2 * np.pi * day_of_year / 365)
df_air_quality_historical["day_of_year_cos"] = np.cos(2 * np.pi * day_of_year / 365)

#### **Tạo biến nhị phân: is_weekend, is_dry_season**

In [ ]:
# 1. Tạo biến is_weekend (Cuối tuần)
df_air_quality_historical["is_weekend"] = (weekday >= 5).astype(int)  # 5: T7 - 6: CN

# 2. Tạo biến is_dry_season (Mùa khô)
# Mùa khô ở TP.HCM thường kéo dài từ tháng 12 đến tháng 4 năm sau
dry_month = [12, 1, 2, 3, 4]
df_air_quality_historical["is_dry_season"] = month.isin(dry_month).astype(int)

### **2.4. Biến mục tiêu**

In [ ]:
# 1. Regression Target: Giá trị AQI (số thực) ngày mai
df_air_quality_historical["target_reg_tomorrow"] = df_air_quality_historical[
    "us_aqi"
].shift(-1)

# 2. Classification Target: Mức độ AQI ngày mai (0=Good, 1=Moderate, ...)
df_air_quality_historical["target_class_tomorrow"] = (
    pd.cut(
        df_air_quality_historical["target_reg_tomorrow"],
        bins=AQI_BINS,
        labels=AQI_LABELS,
        right=True,
    )
    .map(AQI_MAPPING)
    .astype("Int64")  # Dịch nhãn chữ sang số nguyên (0, 1, 2...) bằng AQI_MAPPING
)

# Bỏ tất cả hàng có NaN - do lag/rolling tạo ra ở đầu chuỗi
df_air_quality_historical.dropna(inplace=True)
# Dòng cuối cùng chắc chắn sẽ bị NaN do shift
df_air_quality_historical.dropna(subset=["target_reg_tomorrow"], inplace=True)

print(f"Shape sau khi tạo target: {df_air_quality_historical.shape}")
print(f"\nPhân bố target_class_tomorrow:")
counts = df_air_quality_historical["target_class_tomorrow"].value_counts().sort_index()
for idx, cnt in counts.items():
    pct = cnt / len(df_air_quality_historical) * 100
    print(f"  {int(idx)} - {AQI_LABELS[int(idx)]:<35}: {cnt:4d} ngày ({pct:.1f}%)")

# In ra xem thử kết quả
pd.set_option("display.max_columns", None)
df_air_quality_historical[
    ["us_aqi", "target_reg_tomorrow", "target_class_tomorrow"]
].head(5)


In [ ]:
df_air_quality_historical.describe()

### **2.5. Chia tập Train và Test**

In [ ]:
# 1. Tập X
# Bỏ khỏi X:
## - 'us_aqi'                : TARGET gốc, model không được nhìn thấy khi dự đoán
## - 'target_reg_tomorrow'   : Là y regression
## - 'target_class_tomorrow' : Là y classification
## - 'date'                  : không cần sau khi đã có sin/cos
X = df_air_quality_historical.drop(
    columns=["us_aqi", "target_reg_tomorrow", "target_class_tomorrow", "date"],
    errors="ignore",
)

# 2. Tập y (Gom cả 2 biến mục tiêu của ngày hôm sau)
y = df_air_quality_historical[["target_reg_tomorrow", "target_class_tomorrow"]]

# 3. Chia train test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=False,  # Vì là dữ liệu có tính thời gian nên shuffle = False
)

print(
    f"Train: {len(X_train):,} mẫu | {X_train.index.min().date()} → {X_train.index.max().date()}"
)
print(
    f"Test : {len(X_test):,} mẫu   | {X_test.index.min().date()} → {X_test.index.max().date()}"
)
print(f"Số features tập X: {X_train.shape[1]}")

### **2.6. Xử lý outliers trên tập Train bằng Winsorize**

#### **Đánh giá - Tại sao IQR x 3.0 và không bắt buộc?**

- **Lý do chọn IQR x 3.0 thay vì 1.5:**

Ngưỡng IQR x 1.5 là tiêu chuẩn thống kê thông thường - nhưng với dữ liệu AQI, các ngày có chỉ số cao (130-158) là **sự kiện ô nhiễm thật sự**, không phải lỗi đo lường hay nhập liệu sai. Nếu dùng IQR x 1.5, những ngày này bị clip xuống ngưỡng thấp hơn → model mất đi thông tin quan trọng về các đợt ô nhiễm nặng. IQR x 3.0 chỉ loại bỏ những giá trị **thực sự vô lý về mặt vật lý**, đồng thời giữ lại toàn bộ biến động tự nhiên của chuỗi thời gian.

---

- **Tại sao không bắt buộc với tree-based models:**

Các mô hình XGBoost, LightGBM, Random Forest đưa ra quyết định bằng cách hỏi *"giá trị feature có lớn hơn ngưỡng X không?"* - không tính trung bình, không tính độ lệch chuẩn. Vì vậy một điểm AQI = 158 không làm lệch kết quả như nó làm lệch Linear Regression (kéo đường thẳng về phía nó). Tree-based models **tự nhiên kháng outlier** mà không cần xử lý trước. Ngoài ra, việc dùng `RobustScaler` ở bước chuẩn hóa cũng giảm thiểu thêm ảnh hưởng của outlier vì scaler này dùng median và IQR thay vì mean và std.

---

- **Kết luận:**
> Bước Winsorize trong dự án này có tác dụng chủ yếu với **Ridge Regression** (mô hình baseline tuyến tính). Với XGBoost và LightGBM - hai mô hình chính của dự án - bước này không bắt buộc nhưng được giữ lại để đảm bảo tính nhất quán của pipeline và bảo vệ cho trường hợp so sánh với các mô hình tuyến tính.

In [ ]:
# Winsorize
air_quality_cols = [
    "pm10",
    "pm2_5",
    "carbon_monoxide",
    "nitrogen_dioxide",
    "sulphur_dioxide",
    "ozone",
    "us_aqi",
]
cols_to_winsorize = [col for col in air_quality_cols if col in X_train.columns]
train_fences = {}

# 1. Tính ngưỡng trên Train - tránh Data Leakage từ Test
for col in cols_to_winsorize:
    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)
    IQR = Q3 - Q1
    train_fences[col] = {
        "lower": max(Q1 - 3.0 * IQR, 0),  # AQI không âm
        "upper": Q3 + 3.0 * IQR,
    }

for col in cols_to_winsorize:
    lo = train_fences[col]["lower"]
    up = train_fences[col]["upper"]
    X_train[col] = X_train[col].clip(lo, up)
    X_test[col] = X_test[col].clip(lo, up)

#### **Lý do không xử lý outliers ở các cột còn lại:**
- `date` - là cột thời gian, không phải số. Không có khái niệm outlier cho ngày tháng và không còn dùng cột date nữa vì đã có sin/cos.
- `aerosol_optical_depth` - đây là chỉ số quang học của khí quyển, giá trị tự nhiên dao động rất rộng và bất thường là bình thường - xử lý outlier sẽ làm phức tạp thêm mà không giúp ích cho model.
- `dust` - tương tự, nồng độ bụi sa mạc có thể tăng đột biến khi có gió mùa mang bụi từ xa đến. Spike cao là sự kiện thật, không phải lỗi đo lường.
- `uv_index` - chỉ số tia UV không đưa vào model dự đoán AQI. Có ít ý nghĩa với bài toán ô nhiễm không khí.

In [ ]:
print(
    f"{'Chỉ số':<25} {'Lower':>10} {'Upper':>10} {'Clip Train':>12} {'Clip Test':>11}"
)
print("-" * 72)
for col in cols_to_winsorize:
    lo = train_fences[col]["lower"]
    up = train_fences[col]["upper"]
    n_train = (X_train[col] == lo).sum() + (X_train[col] == up).sum()
    n_test = (X_test[col] == lo).sum() + (X_test[col] == up).sum()
    print(f"{col:<25} {lo:>10.2f} {up:>10.2f} {n_train:>12} {n_test:>11}")

### **2.7. Chuẩn hóa dữ liệu**


In [ ]:
# 1. Xác định lại các cột cần chuẩn hóa (bao gồm cả Lag và Rolling)
# Không scale các biến phân loại/nhị phân (sin, cos, is_weekend, is_dry_season)
exclude_from_scale = [
    "month_sin",
    "month_cos",
    "weekday_sin",
    "weekday_cos",
    "day_of_year_sin",
    "day_of_year_cos",
    "is_weekend",
    "is_dry_season",
]
all_numeric = X_train.select_dtypes(include=["float64", "int64"]).columns.tolist()
cols_to_scale = [c for c in all_numeric if c not in exclude_from_scale]

# 2. Fit trên Train → transform cả hai - KHÔNG fit lại trên Test
scaler = RobustScaler()
X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

print(f"Số cột được scale  : {len(cols_to_scale)}")
print(f"Số cột không scale : {len(exclude_from_scale)} (sin/cos, nhị phân)")
print(f"Tổng features      : {X_train.shape[1]}")
print(f"\nKiểm tra sau scale (mean và median ≈ 0):")
print(X_train[cols_to_scale[:6]].agg(["mean", "median"]).round(3))
display(X_train.head(3))

#### **Kiểm tra tương quan (Correlation Analysis)**
Việc tạo nhiều đặc trưng Lag và Rolling có thể dẫn đến hiện tượng đa cộng tuyến. Cần kiểm tra mối tương quan giữa các đặc trưng và biến mục tiêu `target_reg_tomorrow`.

In [ ]:
# Ghép tạm thời X_train và y_train để xem tương quan
train_full = pd.concat([X_train, y_train["target_reg_tomorrow"]], axis=1)
corr_matrix = train_full.corr()

# Lấy top 15 đặc trưng tương quan nhất với target
top_features = (
    corr_matrix["target_reg_tomorrow"]
    .drop("target_reg_tomorrow", errors="ignore")
    .abs()
    .sort_values(ascending=False)
    .head(15)
)

top_idx = top_features.index.tolist() + ["target_reg_tomorrow"]

plt.figure(figsize=(12, 10))
sns.heatmap(
    train_full[top_idx].corr(),
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8},
)
plt.grid(False)

plt.title("Ma Trận Tương Quan - Top 15 Features với TARGET", fontweight="bold")
plt.show()


print("Top 15 features tương quan nhất với target_reg_tomorrow:")
for feat, val in top_features.items():
    print(f"  {feat:<35} r={val:.3f}")

**Nhận xét:**
- `pm2_5` tương quan cao nhất (r = 0.880) → Là feature quan trọng nhất.
- Rolling mean tương quan cao lẫn nhau → Đa cộng tuyến bình thường, tree-based models xử lý tốt.
- Đồng thời `t-1` cũng là một trong những feature quan trọng, đáng để phân tích vì thường thì chất lượng không khí của những ngày liền kề sẽ có tương quan cao (nhưng r = 0.540).

**Vấn đề có quá nhiều features**

- **Đa cộng tuyến (Multicollinearity):** Nhiều features tương quan cao với nhau.

> Với Linear Regression (Ridge) thì đây là vấn đề thật - model khó phân biệt feature nào thực sự quan trọng, hệ số bị không ổn định.

> Với XGBoost, LightGBM, Random Forest thì không ảnh hưởng - tree-based models split theo ngưỡng, không quan tâm tương quan giữa features.

---

- **Overfitting:** Quá nhiều features so với số mẫu có thể làm model học thuộc dữ liệu Train thay vì học pattern thật.

Dataset: 1.013 mẫu Train

Features hiện tại: 40 features

Tỷ lệ: 1.013 / 40 ≈ 25 mẫu/feature  ← vẫn ổn, ngưỡng nguy hiểm là < 10

> Với 1.013 mẫu và 40 features thì tỷ lệ này vẫn an toàn.

---

Sau khi Train xong, nhìn vào SHAP Summary Plot hoặc Feature Importance - bất kỳ feature nào có importance ≈ 0 thì có thể bỏ đi. Thường thì rolling_max và rolling_min ít quan trọng hơn rolling_mean và rolling_std - có thể bỏ để gọn hơn.

Nhưng với 40 features với 1.013 mẫu là hoàn toàn ổn - XGBoost và LightGBM tự xử lý được. Bỏ feature là bước tùy chọn, không bắt buộc. Nếu thời gian còn nhiều thì làm thêm để thể hiện sự kỹ lưỡng, còn không thì bỏ qua cũng không ảnh hưởng.

> **Định hướng:** Vẫn giữ nguyên toàn bộ các feature ở notebook này thay vào đó sẽ xử lý có chọn lọc ở phần huấn luyện mô hình.

### **2.8. Lưu file**

In [ ]:
outputs_dir = os.path.join(ROOT, "outputs")
models_dir = os.path.join(ROOT, "models")
os.makedirs(outputs_dir, exist_ok=True)
os.makedirs(models_dir, exist_ok=True)

# Lưu Train - Test
save_path_X_train = os.path.join(outputs_dir, "X_train.csv")
save_path_X_test = os.path.join(outputs_dir, "X_test.csv")
save_path_y_train = os.path.join(outputs_dir, "y_train.csv")
save_path_y_test = os.path.join(outputs_dir, "y_test.csv")

X_train.to_csv(save_path_X_train)
X_test.to_csv(save_path_X_test)
y_train.to_csv(save_path_y_train)
y_test.to_csv(save_path_y_test)

print("Đã lưu X_train, X_test, y_train, y_test")

# data_processed.csv - dùng cho Dashboard và predict.py
save_path = os.path.join(outputs_dir, "data_processed.csv")
df_air_quality_historical.to_csv(save_path)
print("Đã lưu data_processed.csv")

# scaler.pkl - bắt buộc dùng đúng scaler đã fit khi predict
save_path = os.path.join(models_dir, "scaler.pkl")
with open(save_path, "wb") as f:
    pickle.dump(scaler, f)
print("Đã lưu scaler.pkl")

# feature_cols.pkl - danh sách features đúng thứ tự, đảm bảo khi dùng predict.py sẽ đúng thứ tự các features
save_path = os.path.join(models_dir, "feature_cols.pkl")
with open(save_path, "wb") as f:
    pickle.dump(X_train.columns.tolist(), f)
print("feature_cols.pkl")

print("\nTổng kết:")
print(f"   X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"   Features: {X_train.columns.tolist()}")

## **2.9. Tổng kết**

Dữ liệu đã sẵn sàng cho bước xây dựng mô hình. Tóm tắt các bước đã thực hiện:

| Bước | Kỹ thuật | Kết quả |
| --- | --- | --- |
| Missing values | `interpolate(method='time', limit_direction='both', inplace=True)` | Điền bù 4 dòng đầu tiên |
| Lag features | `shift(1/2/3/7/14)` | 5 features |
| Rolling features | `shift(1)` + `rolling(3/7/14/30)` | 16 features - không data leakage |
| Sai phân | `diff(1)`, `diff(7)` | 2 features |
| Thời gian | sin/cos encoding | 6 features |
| Nhị phân | `is_weekend`, `is_dry_season` | 2 features |
| TARGET | `shift(-1)` + `pd.cut` | 2 biến mục tiêu |
| Train/Test | 80/20 không shuffle | Train = 1.013, Test = 254 |
| Winsorize | IQR x 3.0 fit trên Train | Không data leakage |
| Chuẩn hóa | `RobustScaler` | Mean ≈ 0, Median ≈ 0 |

#### **Huấn luyện mô hình:** Load `X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv` từ `outputs/`

Target Regression: `target_reg_tomorrow` | Target Classification: `target_class_tomorrow`

---

#### **Cái nhìn tổng quan về dữ liệu trước khi huấn luyện mô hình:**

- Dữ liệu TPHCM có tính ngẫu nhiên cao - AQI ngày hôm nay và ngày mai không tương quan chặt chẽ như các thành phố khác vì:

```text
Đặc điểm:
├── Mưa nhiệt đới đột ngột: AQI có thể giảm 40-50 điểm chỉ sau 1 cơn mưa
├── Gió thay đổi nhanh: Phân tán bụi không đều
├── Không có dữ liệu thời tiết: Nhiệt độ, độ ẩm, tốc độ gió
│   → thiếu các yếu tố quan trọng nhất để dự đoán AQI
└── Chỉ có 5 trạm quan trắc → không đại diện toàn thành phố (những chưa khẳng định được là dữ liệu lấy từ bao nhiêu trạm quan trắc)
```

> **Kết luận:** Dự đoán rằng các kết quả từ việc huấn luyện mô hình sẽ không như mong đợi.